In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "notebook_setup.py").exists()),
    Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebook_setup import get_dataset_path

import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

# ==========================
# ==========================
mat_path = get_dataset_path("CETL-data.mat")
mat_data = sio.loadmat(mat_path, struct_as_record=False, squeeze_me=True)

unbalanced = mat_data["unbalanced_data"]

Vp_cells = unbalanced.Vp
Qp_cells = unbalanced.Qp
Vd_cells = unbalanced.Vd
Qd_cells = unbalanced.Qd

n_cells = len(Qd_cells)

# ==========================
# ==========================
Qmax = np.full(n_cells, np.nan, dtype=float)

for i in range(n_cells):
    qd_mAh = np.array(Qd_cells[i]).astype(float).ravel()
    if qd_mAh.size == 0:
        continue
    qd_Ah = qd_mAh / 1000.0  # mAh → Ah
    Qmax[i] = np.nanmax(qd_Ah)

valid_mask = np.isfinite(Qmax)
valid_Qmax = Qmax[valid_mask]
vmin, vmax = np.min(valid_Qmax), np.max(valid_Qmax)

# ==========================
# ==========================
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 22

fig_width = 20
fig_height = 5

nature_colors = [
    "#313695",
    "#74add1",
    "#e0f3f8",
    "#fee090",
    "#f46d43",
    "#a50026",
]
cmap = LinearSegmentedColormap.from_list("nature_marine", nature_colors, N=256)
norm = Normalize(vmin=vmin, vmax=vmax)

fig, ax = plt.subplots(figsize=(fig_width, fig_height))

V_cut = 2.5

# ==========================
# ==========================
for i in range(n_cells):
    if not valid_mask[i]:
        continue

    vp_mV = np.array(Vp_cells[i]).astype(float).ravel()
    qp_mAh = np.array(Qp_cells[i]).astype(float).ravel()
    if vp_mV.size == 0 or qp_mAh.size == 0:
        continue

    vp = vp_mV / 1000.0   # V
    qp = qp_mAh / 1000.0  # Ah

    n_charge = min(len(vp), len(qp))
    vp = vp[:n_charge]
    qp = qp[:n_charge]

    mask_cut = vp >= V_cut
    if not np.any(mask_cut):
        continue

    start_idx = np.argmax(mask_cut)
    vp_trunc = vp[start_idx:]
    qp_trunc = qp[start_idx:]

    if vp_trunc.size < 2 or qp_trunc.size < 2:
        continue

    qp_norm = qp_trunc - qp_trunc[0]

    vd_mV = np.array(Vd_cells[i]).astype(float).ravel()
    qd_mAh = np.array(Qd_cells[i]).astype(float).ravel()
    if vd_mV.size == 0 or qd_mAh.size == 0:
        continue

    vd = vd_mV / 1000.0   # V
    qd = qd_mAh / 1000.0  # Ah

    n_dis = min(len(vd), len(qd))
    vd = vd[:n_dis]
    qd = qd[:n_dis]

    qd_norm = qd - qd[0]
    qd_shift = qp_norm[-1] + qd_norm

    cap_full = np.concatenate([qp_norm, qd_shift])
    volt_full = np.concatenate([vp_trunc, vd])

    color = cmap(norm(Qmax[i]))
    ax.plot(
        cap_full,
        volt_full,
        color=color,
        linewidth=1.3,
        alpha=0.8
    )

# ==========================
# ==========================
ax.set_xlabel("Relative capacity (Ah)", labelpad=10)
ax.set_ylabel("Voltage (V)", labelpad=20)
ax.set_ylim(2.4, 3.6)
ax.grid(False)

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.015, aspect=10)
cbar.set_label("Maximum capacity (Ah)", labelpad=15)

cbar.outline.set_visible(False)

# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)

fig.tight_layout()

plt.show()
fig.savefig("Charge_discharge_full_cycle_Nature_Marine.png", dpi=300, bbox_inches="tight")